In [0]:
%run ../setup/config

In [0]:
%run ../setup/utils

In [0]:
movies_metadata_df = spark.read.format("delta").load(f"{silver_folder_path}/movies_metadata")
ratings_df = spark.read.format("delta").load(f"{silver_folder_path}/ratings")
links_df = spark.read.format("delta").load(f"{silver_folder_path}/links")
cast_df = spark.read.format("delta").load(f"{silver_folder_path}/cast")

movies_metadata_df.printSchema()
ratings_df.printSchema()
links_df.printSchema()
cast_df.printSchema()


In [0]:
from pyspark.sql import functions as F

final_movies_df = (
  ratings_df
    .join(links_df, links_df.movie_id == ratings_df.movie_id, "inner")
    .join(movies_metadata_df, links_df.tmbd_id == movies_metadata_df.id, "inner")
    .join(cast_df, movies_metadata_df.id == cast_df.id, "inner")
    .groupBy(movies_metadata_df.id, "title", "cast_id", "cast_name", "cast_order")
    .agg(
        F.avg("rating").alias("average_rating"),
        F.count("user_id").alias("number_of_ratings"),
    )
    .filter("number_of_ratings > 400")
    .orderBy(
      F.col("average_rating").desc(),
      F.col("id"),
      F.col("cast_order").asc()
      )
)
display(final_movies_df)

In [0]:
from pyspark.sql.window import Window

df = (
  final_movies_df
    .filter(F.col("cast_order") <= 2)
    .groupBy("cast_id", "cast_name")
    .agg(
      F.countDistinct("id").alias("movies_appeared_in"),
      F.avg("average_rating").alias("avg_movie_rating"),
    )
    .filter("movies_appeared_in >= 8")
    .withColumn(
      "rank",
      F.rank().over(Window.orderBy(F.col("avg_movie_rating").desc())),
    )
    .orderBy("rank")
)

display(df)